# 임베딩 & 벡터 DB 구축

## 전체 파이프라인

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[유저 입력]
  유형 (만화/시리즈/영화/소설/고전)
  + 장르 (스릴러/판타지/로맨스 … 또는 고전 → 국가 → 장르)
  + 소재 (자유 텍스트)
  + 유저 캐릭터 정보 (이름, 성격, 역할 등 — 이미지 없음)
  + AI 캐릭터 정보 (이름, 성격, 외형 등)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[1] 기승전결 뼈대 생성  (1회)
  RAG: 기/승/전/결 단계별 유사 씬 검색 (← 이 파일이 담당)
  → LLM: 각 단계의 핵심 사건 방향성만 생성
          (구체적 대본 X, 결말 고정 X)
  출력 예시
    기) 형사가 연쇄 실종 사건을 맡게 됨
    승) 단서를 쫓다 배후 조직의 표적이 됨
    전) 동료의 배신으로 고립된 위기에 처함
    결) 유저의 선택에 따라 열린 결말
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[2] 캐릭터 생성  (1회)
  ┌─ 유저 캐릭터
  │   유저 입력 기반 텍스트 페르소나 생성
  │   (이미지 없음 — 이야기를 이끄는 플레이어 역할)
  │
  └─ AI 캐릭터
      유저 입력 기반 텍스트 페르소나 생성
      + 외형 설명 저장 (이미지 일관성 유지용)
      + 초기 이미지 생성 (캐릭터 소개 이미지)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[3] 인터랙티브 채팅  (반복)
  유저 캐릭터 ↔ AI 캐릭터 자유 대화
  시스템이 현재 단계 추적: 기 → 승 → 전 → 결
  유저의 대화 내용에 따라 분기 자동 결정
  → 다음 단계 진입 시 RAG로 해당 단계 씬 추가 참고
  → 단계 전환 시점마다 이미지 생성
       (저장된 AI 캐릭터 외형 + 현재 상황 반영)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
          ↓
[4] 결말 도달
  결 단계 완료
  → 유저의 대화 흐름에 따라 결정된 결말 출력
  → 결말 이미지 생성 (AI 캐릭터 외형 + 결말 상황 반영)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[이미지 생성 흐름]
  AI 캐릭터 외형 설명 (텍스트, 1회 저장)
          ↓
  캐릭터 소개 이미지 ── 기 진입 이미지 ── 승 진입 이미지
                                              ↓
                         결 결말 이미지 ── 전 진입 이미지
  → 매번 동일한 외형 설명 + 해당 단계 상황 설명을 합쳐서 생성
```

## 이 파일의 범위

| 단계 | 이 파일 | 미구현 |
|------|---------|--------|
| RAG 벡터 DB 구축 | ✅ | |
| 유형/장르/소재 기반 씬 검색 | ✅ | |
| 고전 탭 국가/장르 필터 검색 | ✅ | |
| RAG 컨텍스트 텍스트 조립 | ✅ | |
| 기승전결 뼈대 생성 프롬프트 | | ❌ |
| 유저 캐릭터 생성 프롬프트 | | ❌ |
| AI 캐릭터 생성 프롬프트 | | ❌ |
| AI 캐릭터 외형 저장 & 일관성 관리 | | ❌ |
| 채팅 단계 추적 & 전환 로직 | | ❌ |
| 단계별 이미지 생성 | | ❌ |
| 결말 생성 & 결말 이미지 생성 | | ❌ |

## 설계 결정 사항

| 항목 | 결정 | 근거 |
|------|------|------|
| **임베딩 모델** | `text-embedding-3-small` | 비용 효율적 ($0.02/1M tokens) |
| **씬 임베딩 텍스트** | `unit_motif + storyline` | 의미적 유사도 검색에 최적화 |
| **고전 임베딩 텍스트** | `주제문` | 단락 내용 요약 1문장 |
| **category_name** | 메타데이터 필터 | 만화/시리즈/영화/소설 유형 구분 |
| **stage** | 메타데이터 필터 | 기승전결 단계별 씬 검색 |
| **causality** | 임베딩 제외, 프롬프트 출력용 | 씬 간 연결 정보 |
| **hero_journey** | framework 필터 없이 유사도에 맡김 | 판타지 맥락 시 자연스럽게 포함 |
| **고전 데이터** | 고전 탭 선택 시에만 추가 검색 | 세계관·분위기 보강 보조 데이터 |

## 출력 파일

| 경로 | 내용 |
|------|------|
| `vectordb/scenes` | 문화콘텐츠 씬 컬렉션 (89,851개) |
| `vectordb/classics` | 동아시아 고전 단락 컬렉션 (12,717개) |

In [17]:
# 필요 패키지 설치
#!pip install openai chromadb tqdm

In [18]:
import json
import os
import time
from pathlib import Path
from typing import Optional

import chromadb
from openai import OpenAI
from tqdm import tqdm
from dotenv import load_dotenv 

load_dotenv()

# ── 경로 설정 ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("C:/Users/User/Desktop/Github/Dive.ai/output")
DB_DIR     = Path("C:/Users/User/Desktop/Github/Dive.ai/vectordb")
DB_DIR.mkdir(exist_ok=True)

# ── OpenAI 클라이언트 ──────────────────────────────────────────────────────────
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

EMBED_MODEL = "text-embedding-3-small"  # 1536차원
BATCH_SIZE  = 500                        # OpenAI API 배치 크기
UPSERT_BATCH = 100                       # ChromaDB 업서트 배치 크기

print(f"OUTPUT_DIR 존재 여부: {OUTPUT_DIR.exists()}")
print(f"DB_DIR: {DB_DIR}")
if os.environ.get("OPENAI_API_KEY"):
    print(f"API Key 로드 완료: {os.environ.get('OPENAI_API_KEY')[:7]}...")
else:
    print("API Key를 찾을 수 없습니다. .env 파일을 확인해주세요.")

OUTPUT_DIR 존재 여부: True
DB_DIR: C:\Users\User\Desktop\Github\Dive.ai\vectordb
API Key 로드 완료: sk-proj...


## 1. 전처리 데이터 로드

`문화콘텐츠,고전_전처리.ipynb` 실행 결과로 생성된 JSON 파일을 로드합니다.

In [19]:
with open(OUTPUT_DIR / "scene_metadata_index.json", encoding="utf-8") as f:
    scene_index = json.load(f)

with open(OUTPUT_DIR / "classic_paragraph_index.json", encoding="utf-8") as f:
    classic_index = json.load(f)

print(f"문화콘텐츠 씬:   {len(scene_index):,}개")
print(f"동아시아 고전 단락: {len(classic_index):,}개")

문화콘텐츠 씬:   89,851개
동아시아 고전 단락: 12,717개


## 2. 임베딩 텍스트 구성

### 씬 임베딩 텍스트
- `unit_motif + storyline` 조합
- `unit_motif` 누락 시 `stage`로 fallback (7.4% 누락 대응)
- `causality`는 임베딩 제외 — 씬 자체 내용이 아닌 다음 씬으로의 연결 정보

### 고전 단락 임베딩 텍스트
- `주제문` 단독 사용 — 단락 내용을 한 문장으로 요약한 필드

In [20]:
def build_scene_text(scene: dict) -> str:
    motif     = scene.get("unit_motif") or scene.get("stage", "")
    storyline = scene.get("storyline", "")
    return f"{motif} {storyline}".strip()


def build_classic_text(para: dict) -> str:
    return para.get("summary", "")


# ── 임베딩 비용 예측 ───────────────────────────────────────────────────────────
AVG_SCENE_TOKENS   = 35   # unit_motif + storyline 평균 토큰 수 추정
AVG_CLASSIC_TOKENS = 45   # 주제문 평균 토큰 수 추정
COST_PER_1M        = 0.02 # text-embedding-3-small (USD)

total_tokens = len(scene_index) * AVG_SCENE_TOKENS + len(classic_index) * AVG_CLASSIC_TOKENS
estimated_cost = total_tokens / 1_000_000 * COST_PER_1M

print(f"예상 총 토큰:  {total_tokens:,}")
print(f"예상 비용:     ${estimated_cost:.4f} (text-embedding-3-small 기준)")
print()

# 샘플 확인
sample = scene_index[0]
print("[씬 임베딩 텍스트 샘플]")
print(build_scene_text(sample))
print()
print("[고전 임베딩 텍스트 샘플]")
print(build_classic_text(classic_index[0]))

예상 총 토큰:  3,717,050
예상 비용:     $0.0743 (text-embedding-3-small 기준)

[씬 임베딩 텍스트 샘플]
긴장감 C001은 지하철 안에서 졸다가 깨어 내리는데 두 여자아이를 발견하고 섬찟한다.

[고전 임베딩 텍스트 샘플]
연왕후가 모란을 죽이기 위해 궁녀를 병사로 삼겠다고 하고 상께서 궁녀를 모았지만 그 중에 모란이 보이지 않았다.


## 3. ChromaDB 컬렉션 초기화

- `scenes`: 문화콘텐츠 씬 컬렉션
- `classics`: 동아시아 고전 단락 컬렉션
- 거리 함수: cosine (유사도 = 1 - distance)
- PersistentClient: `vectordb/` 폴더에 영구 저장

In [21]:
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))

scene_collection = chroma_client.get_or_create_collection(
    name="scenes",
    metadata={"hnsw:space": "cosine"},
)

classic_collection = chroma_client.get_or_create_collection(
    name="classics",
    metadata={"hnsw:space": "cosine"},
)

print(f"scenes  컬렉션 현재 항목 수: {scene_collection.count():,}")
print(f"classics 컬렉션 현재 항목 수: {classic_collection.count():,}")

scenes  컬렉션 현재 항목 수: 89,851
classics 컬렉션 현재 항목 수: 12,717


## 4. 문화콘텐츠 씬 임베딩 및 저장

### 저장 메타데이터

| 필드 | 용도 |
|------|------|
| `work_id` | 작품 식별자 |
| `category_name` | 유형 필터 (만화/시리즈/영화/소설) |
| `stage` | 기승전결 파트 필터링 |
| `framework` | storyhelper / hero_journey 필터링 |
| `genre` | 장르 post-filtering (콤마 구분 문자열) |
| `theme` | 작품 테마 (출력 컨텍스트) |
| `unit_motif` | 검색 결과 출력 컨텍스트 |
| `work_motif` | 작품 모티프 참고 |
| `main_character` | 주인공 유형 참고 |
| `storyline` | LLM 프롬프트 주입 텍스트 |
| `causality` | 있을 때만 프롬프트에 포함 (씬 간 인과 힌트) |
| `dominant_emotions` | 저장만, 현재 미사용 |

In [22]:
def embed_and_store_scenes():
    current_count = scene_collection.count()
    total_count   = len(scene_index)

    if current_count >= total_count:
        print(f"이미 완료: {current_count:,}개 씬 저장됨. 스킵합니다.")
        return

    print("기존 저장 ID 확인 중...")
    existing_ids = set()
    if current_count > 0:
        result = scene_collection.get(include=[])
        existing_ids = set(result["ids"])

    to_embed = [s for s in scene_index if s["scene_id"] not in existing_ids]
    print(f"임베딩 대상: {len(to_embed):,}개 / 전체 {total_count:,}개")

    if not to_embed:
        print("임베딩할 씬이 없습니다.")
        return

    texts = [build_scene_text(s) for s in to_embed]
    embeddings = []

    print("OpenAI 임베딩 중...")
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i:i + BATCH_SIZE]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        embeddings.extend([r.embedding for r in response.data])
        time.sleep(0.05)

    print("ChromaDB 저장 중...")
    for i in tqdm(range(0, len(to_embed), UPSERT_BATCH)):
        batch_scenes     = to_embed[i:i + UPSERT_BATCH]
        batch_embeddings = embeddings[i:i + UPSERT_BATCH]

        scene_collection.upsert(
            ids=[s["scene_id"] for s in batch_scenes],
            embeddings=batch_embeddings,
            documents=[build_scene_text(s) for s in batch_scenes],
            metadatas=[
                {
                    "work_id":           s.get("work_id", ""),
                    "category_name":     s.get("category_name", ""),   # 만화/시리즈/영화/소설
                    "genre":             ",".join(s.get("genre", [])),
                    "stage":             s.get("stage", ""),
                    "framework":         s.get("framework", "storyhelper"),
                    "theme":             s.get("theme", ""),
                    "unit_motif":        s.get("unit_motif") or s.get("stage", ""),
                    "work_motif":        s.get("work_motif", ""),
                    "main_character":    s.get("main_character", ""),
                    "storyline":         (s.get("storyline") or "")[:500],
                    "causality":         (s.get("causality") or "")[:300],
                    "dominant_emotions": ",".join(s.get("dominant_emotions", [])),
                }
                for s in batch_scenes
            ],
        )

    print(f"완료. 총 {scene_collection.count():,}개 씬 저장됨.")


embed_and_store_scenes()

이미 완료: 89,851개 씬 저장됨. 스킵합니다.


## 5. 동아시아 고전 단락 임베딩 및 저장

고전 데이터는 **고전 배경 선택 시에만** 세계관·분위기 보강용으로 참조합니다.  
`주제문` 한 문장을 임베딩하여 유저 입력과의 의미적 유사도를 계산합니다.

In [23]:
# 기존에 잘못 저장된 (메타데이터가 빈) 고전 컬렉션을 삭제합니다.
chroma_client.delete_collection("classics")
# 다시 새로 만듭니다.
classic_collection = chroma_client.get_or_create_collection(
name="classics",
metadata={"hnsw:space": "cosine"},
)


def embed_and_store_classics():
    current_count = classic_collection.count()
    total_count   = len(classic_index)

    if current_count >= total_count:
        print(f"이미 완료: {current_count:,}개 단락 저장됨. 스킵합니다.")
        return

    print("기존 저장 ID 확인 중...")
    existing_ids = set()
    if current_count > 0:
        result = classic_collection.get(include=[])
        existing_ids = set(result["ids"])

    to_embed = [p for p in classic_index if p["paragraph_id"] not in existing_ids]
    print(f"임베딩 대상: {len(to_embed):,}개 / 전체 {total_count:,}개")

    if not to_embed:
        print("임베딩할 단락이 없습니다.")
        return

    texts = [build_classic_text(p) for p in to_embed]
    embeddings = []

    print("OpenAI 임베딩 중...")
    for i in tqdm(range(0, len(texts), BATCH_SIZE)):
        batch = texts[i:i + BATCH_SIZE]
        response = client.embeddings.create(model=EMBED_MODEL, input=batch)
        embeddings.extend([r.embedding for r in response.data])
        time.sleep(0.05)

    print("ChromaDB 저장 중...")
    for i in tqdm(range(0, len(to_embed), UPSERT_BATCH)):
        batch_paras      = to_embed[i:i + UPSERT_BATCH]
        batch_embeddings = embeddings[i:i + UPSERT_BATCH]

        classic_collection.upsert(
            ids=[p["paragraph_id"] for p in batch_paras],
            embeddings=batch_embeddings,
            documents=[build_classic_text(p) for p in batch_paras],
            metadatas=[
                {
                    "work_id":   p.get("work_id", ""),
                    "country":   p.get("country", ""),
                    "genre":     ",".join(p.get("genre", [])),
                    "motif":     ",".join(p.get("motif", [])),
                    "space":     ",".join(p.get("space", [])),
                    "character": ",".join(p.get("character", [])),
                    "summary":    (p.get("summary") or "")[:500],
                }
                for p in batch_paras
            ],
        )

    print(f"완료. 총 {classic_collection.count():,}개 고전 단락 저장됨.")


embed_and_store_classics()

기존 저장 ID 확인 중...
임베딩 대상: 12,717개 / 전체 12,717개
OpenAI 임베딩 중...


100%|██████████| 26/26 [00:57<00:00,  2.21s/it]


ChromaDB 저장 중...


100%|██████████| 128/128 [00:36<00:00,  3.52it/s]

완료. 총 12,717개 고전 단락 저장됨.


## 6. 기승전결 Stage 매핑

스토리헬퍼 16단계를 기승전결 4구간으로 묶어 `story_arc` 파라미터로 검색할 수 있도록 합니다.  
stage는 임베딩에 포함되지 않고 **메타데이터 post-filtering**에만 사용됩니다.

In [24]:
# 스토리헬퍼 16단계 → 기승전결 매핑
STORY_ARC_STAGES: dict[str, list[str]] = {
    "기": [
        "Opening Salvo",    # 첫 장면, 세계관 제시
        "Main Character",   # 주인공 소개
        "Setting-up",       # 배경 및 관계 설정
    ],
    "승": [
        "1st Accident",     # 첫 번째 사건
        "Villains Move",    # 적대자 등장
        "Doubts & Debate",  # 주인공 갈등
        "Making a Choice",  # 결단
        "Choice to Fight",  # 싸우기로 결심
    ],
    "전": [
        "Ups & Downs",      # 갈등 반복
        "2nd Accident",     # 두 번째 위기
        "Innermost Cave",   # 최대 위험
        "Defeat",           # 패배/최대 위기
        "Resurrection",     # 재기
        "Another Story",    # 서브플롯
    ],
    "결": [
        "Trailer Moments",  # 클라이맥스 직전
        "Final Salvo",      # 최종 해결
    ],
}

for arc, stages in STORY_ARC_STAGES.items():
    print(f"{arc}: {', '.join(stages)}")

기: Opening Salvo, Main Character, Setting-up
승: 1st Accident, Villains Move, Doubts & Debate, Making a Choice, Choice to Fight
전: Ups & Downs, 2nd Accident, Innermost Cave, Defeat, Resurrection, Another Story
결: Trailer Moments, Final Salvo


## 7. 검색 함수

### retrieve_scenes
- 유저 입력을 임베딩 → 유사 씬 검색
- `genre`, `story_arc`, `stage`, `framework` 필터 지원
- genre는 ChromaDB 메타데이터 substring 미지원으로 **Python post-filtering**
- `framework=None`: storyhelper + hero_journey 전체 검색 (판타지 기본)

### retrieve_classics
- 고전 배경 선택 시에만 호출
- `country` 필터 + genre/motif post-filtering

In [25]:
def retrieve_scenes(
    query: str,
    category: Optional[str] = None,
    genre: Optional[str] = None,
    stage: Optional[str] = None,
    story_arc: Optional[str] = None,
    framework: Optional[str] = None,
    top_k: int = 5,
) -> list[dict]:
    query_vec = client.embeddings.create(
        model=EMBED_MODEL, input=[query]
    ).data[0].embedding

    # 1. ChromaDB where 필터 구성 (DB 레벨에서 먼저 거름)
    where = {}
    if framework:
        where["framework"] = {"$eq": framework}
    if category:
        where["category_name"] = {"$eq": category}

    # [수정] 기승전결(story_arc) 필터를 DB 검색 조건($in)에 직접 포함
    arc_stages = STORY_ARC_STAGES.get(story_arc, []) if story_arc else []

    if stage:
        where["stage"] = {"$eq": stage}
    elif arc_stages:
        where["stage"] = {"$in": arc_stages} # 해당 단계인 것만 검색

    # [수정] 장르 필터링을 위해 fetch_k를 훨씬 넉넉하게 잡음 (15배)
    fetch_k = top_k * 15 if genre else top_k
    fetch_k = min(fetch_k, scene_collection.count())

    query_params: dict = {
        "query_embeddings": [query_vec],
        "n_results":        fetch_k,
        "include":          ["metadatas", "documents", "distances"],
    }
    if where:
        query_params["where"] = where

    results = scene_collection.query(**query_params)

    scenes = []
    for meta, doc, dist in zip(
        results["metadatas"][0],
        results["documents"][0],
        results["distances"][0],
    ):
        # 장르는 콤마로 구분된 문자열이므로 파이썬에서 최종 필터링
        if genre and genre not in meta.get("genre", ""):
            continue

        scenes.append({
            "score":         round(1 - dist, 4),
            "category":      meta.get("category_name", ""),
            "stage":         meta.get("stage", ""),
            "framework":     meta.get("framework", ""),
            "unit_motif":    meta.get("unit_motif", ""),
            "genre":         meta.get("genre", ""),
            "storyline":     meta.get("storyline", ""),
            "causality":     meta.get("causality", ""),
        })
        if len(scenes) >= top_k:
            break

    return scenes


def retrieve_classics(
    query: str,
    country: Optional[str] = None,
    genre_keywords: Optional[list[str]] = None,
    motif_keywords: Optional[list[str]] = None,
    top_k: int = 3,
) -> list[dict]:
    query_vec = client.embeddings.create(
        model=EMBED_MODEL, input=[query]
    ).data[0].embedding

    where = {}
    if country:
        where["country"] = {"$eq": country}

    fetch_k = top_k * 10 if (genre_keywords or motif_keywords) else top_k
    fetch_k = min(fetch_k, classic_collection.count())

    query_params: dict = {
        "query_embeddings": [query_vec],
        "n_results":        fetch_k,
        "include":          ["metadatas", "documents", "distances"],
    }
    if where:
        query_params["where"] = where

    results = classic_collection.query(**query_params)

    paragraphs = []
    for meta, doc, dist in zip(
        results["metadatas"][0],
        results["documents"][0],
        results["distances"][0],
    ):
        if genre_keywords and not any(kw in meta.get("genre", "") for kw in genre_keywords):
            continue
        if motif_keywords and not any(kw in meta.get("motif", "") for kw in motif_keywords):
            continue

        paragraphs.append({
            "score":   round(1 - dist, 4),
            "country": meta.get("country", ""),
            "genre":   meta.get("genre", ""),
            "motif":   meta.get("motif", ""),
            "space":   meta.get("space", ""),
            "summary": meta.get("summary", ""), # [수정] 주제문 -> summary
        })
        if len(paragraphs) >= top_k:
            break

    return paragraphs


print("검색 함수 정의 완료.")

검색 함수 정의 완료.


## 8. RAG 프롬프트 빌더

검색 결과를 LLM 프롬프트 주입용 텍스트로 변환합니다.

### 유형별 선택 가능 장르 (`CATEGORY_TO_GENRES`)

소설 카테고리에 `공포(호러)` 데이터가 없으므로, 소설 선택 시 해당 장르를 프론트엔드에서 노출하지 않습니다.

| 유형 | 장르 선택지 |
|------|------------|
| 영화 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |
| 시리즈 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |
| 소설 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 전쟁 **(공포(호러) 제외)** |
| 만화 | 드라마 · 멜로/로맨스 · 스릴러 · 판타지 · 액션 · 미스터리 · 코미디 · SF · 공포(호러) · 전쟁 |

### 고전 탭 UI → 백엔드 파라미터 흐름

```
[프론트엔드]                         [백엔드]
장르 탭 선택
  └─ 일반 장르 선택 (스릴러 등)  →  genre="스릴러", country=None
  └─ 고전 탭
       └─ 한국 선택              →  country="한국"
            └─ 가문소설 선택     →  classic_genre="가문소설"
            └─ (미선택)          →  classic_genre=None  ← 국가 전체 검색
       └─ 중국 선택              →  country="중국"
       └─ 일본 선택              →  country="일본"
```

### 국가별 선택 가능 장르 (`COUNTRY_TO_GENRES`)

| 국가 | 장르 선택지 |
|------|------------|
| 한국 | 가문소설 · 판타지 · 로맨스 · 영웅소설 · 미스터리 · 호러 |
| 중국 | 판타지 · 로맨스 · 무협 · 호러 · 미스터리 |
| 일본 | 설화 · 미스터리 · 호러 · 편지소설 · 영험담 |

### 동작 방식
- `country=None` → 고전 단락 검색 안 함 (일반 씬만 사용)
- `classic_genre=None` → 해당 국가 전체 장르 대상으로 검색
- `classic_genre="설화"` → 해당 장르 단일 필터로 검색
- `causality`: 있을 때만 씬 컨텍스트에 포함 — 씬 간 흐름 힌트

In [26]:
# ── 유형별 장르 목록 (프론트엔드 선택지 관리) ─────────────────────────────────
# 소설 카테고리에 공포(호러) 데이터 없음 → 소설 선택 시 해당 장르 노출 안 함
CATEGORY_TO_GENRES: dict[str, list[str]] = {
    "영화":   ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "공포(호러)", "전쟁"],
    "시리즈": ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "공포(호러)", "전쟁"],
    "소설":   ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "전쟁"],
    "만화":   ["드라마", "멜로/로맨스", "스릴러", "판타지", "액션", "미스터리", "코미디", "SF", "공포(호러)", "전쟁"],
}

# ── 국가별 고전 장르 목록 (프론트엔드 선택지 + 백엔드 필터 공유) ───────────────
# 실제 데이터 기준 (동아시아 고전.ipynb 섹션 6 참고)
# 한국: 가문소설(3,545) · 판타지(2,655) · 로맨스(1,747) · 영웅소설(380) 등
# 중국: 판타지(1,137) · 로맨스(204) · 무협(194) · 호러(149) 등
# 일본: 설화(2,980) · 미스터리(235) · 호러(82) · 편지소설(72) 등

COUNTRY_TO_GENRES: dict[str, list[str]] = {
    "한국": ["가문소설", "판타지", "로맨스", "영웅소설", "미스터리", "호러"],
    "중국": ["판타지", "로맨스", "무협", "호러", "미스터리"],
    "일본": ["설화", "미스터리", "호러", "편지소설", "영험담"],
}

CLASSIC_COUNTRIES: set[str] = set(COUNTRY_TO_GENRES.keys())  # {"한국", "중국", "일본"}


def build_scene_context(scenes: list[dict]) -> str:
    if not scenes:
        return ""
    lines = ["[유사 작품 씬 패턴 참고]"]
    for s in scenes:
        tag  = f"{s['stage']} / {s['unit_motif']}" if s["unit_motif"] else s["stage"]
        line = f"- [{tag}] {s['storyline']}"
        if s.get("causality"):
            line += f"\n  → {s['causality']}"
        lines.append(line)
    return "\n".join(lines)


def build_classic_context(paragraphs: list[dict]) -> str:
    if not paragraphs:
        return ""
    lines = ["[동아시아 고전 참고 — 세계관·분위기]"]
    for p in paragraphs:
        motif_str = p["motif"][:40] if p["motif"] else "?"
        space_str = p["space"][:20] if p["space"] else "?"
        lines.append(f"- [{p['country']} / {motif_str} / {space_str}] {p['summary']}")
    return "\n".join(lines)


def build_rag_context(
    query: str,
    category: Optional[str] = None,
    genre: Optional[str] = None,
    story_arc: Optional[str] = None,
    country: Optional[str] = None,
    classic_genre: Optional[str] = None,
    top_k_scenes: int = 5,
    top_k_classics: int = 3,
) -> str:
    """씬 + 고전 RAG 컨텍스트 통합 생성.

    Parameters
    ----------
    query         : 유저 입력 소재/맥락
    category      : 유형 ('만화' | '시리즈' | '영화' | '소설') — 고전 탭이면 None
    genre         : 장르 (예: '스릴러') — CATEGORY_TO_GENRES 기준으로 프론트에서 검증됨
    story_arc     : 기승전결 파트 (예: '결')
    country       : 고전 탭 선택 국가 ('한국' | '중국' | '일본') — None이면 고전 미사용
    classic_genre : 국가 내 선택 장르 (예: '가문소설') — None이면 국가 전체 검색
    top_k_scenes  : 씬 검색 수
    top_k_classics: 고전 단락 검색 수
    """
    # 씬 검색 — framework=None: storyhelper + hero_journey 전체 (유사도에 맡김)
    scenes  = retrieve_scenes(
        query, category=category, genre=genre, story_arc=story_arc, top_k=top_k_scenes
    )
    context = build_scene_context(scenes)

    # 고전 탭 선택 시에만 고전 단락 추가
    if country and country in CLASSIC_COUNTRIES:
        genre_keywords = [classic_genre] if classic_genre else COUNTRY_TO_GENRES.get(country)
        classics = retrieve_classics(
            query,
            country=country,
            genre_keywords=genre_keywords,
            top_k=top_k_classics,
        )
        if classics:
            context += "\n\n" + build_classic_context(classics)

    return context


print("RAG 프롬프트 빌더 정의 완료.")
print()
print("CATEGORY_TO_GENRES:")
for cat, genres in CATEGORY_TO_GENRES.items():
    print(f"  {cat}: {', '.join(genres)}")
print()
print("COUNTRY_TO_GENRES:")
for c, genres in COUNTRY_TO_GENRES.items():
    print(f"  {c}: {', '.join(genres)}")

RAG 프롬프트 빌더 정의 완료.

CATEGORY_TO_GENRES:
  영화: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 공포(호러), 전쟁
  시리즈: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 공포(호러), 전쟁
  소설: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 전쟁
  만화: 드라마, 멜로/로맨스, 스릴러, 판타지, 액션, 미스터리, 코미디, SF, 공포(호러), 전쟁

COUNTRY_TO_GENRES:
  한국: 가문소설, 판타지, 로맨스, 영웅소설, 미스터리, 호러
  중국: 판타지, 로맨스, 무협, 호러, 미스터리
  일본: 설화, 미스터리, 호러, 편지소설, 영험담


## 9. 검색 테스트

임베딩 완료 후 아래 셀을 실행하여 검색 품질을 확인합니다.

In [27]:
SEP = "=" * 60

# ── 테스트 1: 스릴러 — 승 파트 (고전 미선택) ──────────────────────────────────
print(SEP)
print("테스트 1: 스릴러 — 승 파트 (고전 미선택)")
print(SEP)
context = build_rag_context(
    query="형사가 사건의 단서를 쫓으며 위험에 빠진다",
    genre="스릴러",
    story_arc="승",
)
print(context)

print()

# ── 테스트 2: 판타지 — 결 파트 (고전 미선택) ──────────────────────────────────
print(SEP)
print("테스트 2: 판타지 — 결 파트 (고전 미선택)")
print(SEP)
context = build_rag_context(
    query="모든 시련을 이겨낸 영웅이 고향으로 돌아온다",
    genre="판타지",
    story_arc="결",
)
print(context)

print()

# ── 테스트 3: 고전 탭 → 한국 → 가문소설 ─────────────────────────────────────
print(SEP)
print("테스트 3: 고전 탭 → 한국 → 가문소설")
print(SEP)
context = build_rag_context(
    query="두 가문의 갈등 속에서 피어나는 비극적 사랑 이야기",
    genre="로맨스",
    story_arc="전",
    country="한국",
    classic_genre="가문소설",
)
print(context)

print()

# ── 테스트 4: 고전 탭 → 중국 → 무협 ─────────────────────────────────────────
print(SEP)
print("테스트 4: 고전 탭 → 중국 → 무협")
print(SEP)
context = build_rag_context(
    query="강호를 떠돌며 복수를 꿈꾸는 무사의 여정",
    genre="판타지",
    story_arc="승",
    country="중국",
    classic_genre="무협",
)
print(context)

print()

# ── 테스트 5: 고전 탭 → 일본 → 설화 ─────────────────────────────────────────
print(SEP)
print("테스트 5: 고전 탭 → 일본 → 설화")
print(SEP)
context = build_rag_context(
    query="요괴를 물리치고 마을을 구하는 승려의 이야기",
    genre="판타지",
    story_arc="결",
    country="일본",
    classic_genre="설화",
)
print(context)

print()

# ── 테스트 6: 고전 탭 → 한국 (장르 미선택 — 국가 전체 검색) ──────────────────
print(SEP)
print("테스트 6: 고전 탭 → 한국 (장르 미선택)")
print(SEP)
context = build_rag_context(
    query="조선 시대 신분을 초월한 사랑과 시련",
    story_arc="기",
    country="한국",
    classic_genre=None,  # 국가 전체 장르 검색
)
print(context)

테스트 1: 스릴러 — 승 파트 (고전 미선택)
[유사 작품 씬 패턴 참고]
- [1st Accident / 상황 설명] 이 형사가 피해자들의 신원조회 내용을 보고한다.
  → Seeker(추적/추구)
- [Making a Choice / 감정을 부정] C002은 너무 힘들어 자살을 시도하고 판사는 C003에게 사형을 선고한다.
  → Lack(결핍)
- [1st Accident / 신분위장] 대신 안전하게 지켜주겠다며 형사는 수녀원에서 두 달만 숨어 있으라고 한다.
  → Crisis of the heart(지극한 슬픔)
- [Making a Choice / 강제된 과업] C001가 고용한 탐정이 죽고 위험에 빠진다.
  → Facing the greatest fear(심연을 직면함)
- [Doubts & Debate / 사고사] 소방대원들이 죽어나가고, 하객들도 위험에 처한다.
  → Lack(결핍)

테스트 2: 판타지 — 결 파트 (고전 미선택)
[유사 작품 씬 패턴 참고]
- [Final Salvo / 덜떨어진 영웅] C018는 마을의 영웅이 되고 주민들이 무너진 C012의 집을 복구해준다.
  → Courtship(구애)
- [Final Salvo / Final Salvo] 모두들 현실로 돌아와서 각자 돌아가지만 놀이기구에서의 기억들은 다들 아름답게 가져간다.
- [Final Salvo / 이별] C001는 C012과 야생마들을 자연으로 돌려보낸다.
  → Return(귀환)
- [Final Salvo / 시련] 생명력이 바닥난 C001는 몰려오는 적들을 보고 죽음을 받아들인다.
  → Last test(마지막 시험/시련)
- [Final Salvo / 해피엔딩] 평온을 되찾은 탐사대원들은 후손을 이어가며 항해를 계속한다.
  → Return(귀환)

테스트 3: 고전 탭 → 한국 → 가문소설
[유사 작품 씬 패턴 참고]
- [Resurrection / 광기어린 사랑] 두 사람이 배를 향해 헤엄쳐간다.
  → Setback(후퇴/역행/좌절)
- [A